In [1]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

### Sample Submission

Load the training data:

In [2]:
train_df = pd.read_csv('data/train.csv')
train_df.head()

,Title,Description,location,Carpet Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area,Price_USD,Amount_USD
0,3 BHK Ready to Occupy Flat for sale in Runwal ...,Carefully laid out in the prime location of Gh...,mumbai,806 sqft,Ready to Move,Upper Basement out of 15,Resale,Semi-Furnished,East,Main Road,Runwal Orchard Residency,2,3,1 Covered,Freehold,NaN,NaN,NaN,199.0 USD,271084.0
1,2 BHK Ready to Occupy Flat for sale in Shilale...,Up for immediate sale is a 2 BHK apartment in ...,ahmedabad,900 sqft,Ready to Move,1 out of 4,Resale,Unfurnished,NaN,Main Road,Shilalekh Apartments,2,2,NaN,Freehold,NaN,NaN,NaN,70.0 USD,63012.0
2,2 BHK Ready to Occupy Flat for sale Koramangala,This magnificent 2 BHK Flat is available for s...,bangalore,1000 sqft,Ready to Move,Ground out of 7,New Property,Unfurnished,West,Garden/Park,NaN,2,1,1 Covered,Freehold,NaN,NaN,NaN,127.0 USD,186747.0
3,4 BHK Ready to Occupy Flat for sale in Hiranan...,Have a look at this immaculate 4 BHK flat for ...,thane,1520 sqft,Ready to Move,14 out of 27,Resale,Unfurnished,East,Garden/Park,Hiranandani Meadows,4,1,2 Covered,Freehold,NaN,NaN,NaN,144.0 USD,289157.0
4,3 BHK Ready to Occupy Flat for sale Jagatpur,Have a look at this immaculate 3 BHK flat for ...,ahmedabad,NaN,Ready to Move,3 out of 9,Resale,Unfurnished,NaN,NaN,NaN,3,NaN,NaN,NaN,1773 sqft,NaN,NaN,50.0 USD,90361.0


Load the testing data:

In [3]:
test_df = pd.read_csv('data/test_inputs.csv')
test_df.head()

,Title,Description,location,Carpet Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area,Price_USD
0,3 BHK Ready to Occupy Flat for sale in Bhartiy...,"Thanisandra Main Road, Bangalore has an appeal...",bangalore,1415 sqft,Ready to Move,32 out of 32,Resale,Semi-Furnished,East,NaN,NaN,3,3,NaN,NaN,NaN,NaN,NaN,157.0 USD
1,3 BHK Ready to Occupy Flat for sale in DS Max ...,This ready to move-in 3 BHK flat is available ...,bangalore,NaN,Ready to Move,5 out of 5,Resale,Semi-Furnished,NaN,NaN,NaN,4,NaN,NaN,NaN,3350 sqft,NaN,NaN,46.0 USD
2,2 BHK Ready to Occupy Flat for sale in Vijay E...,2 BHK flat available for sale in Thane in the ...,thane,800 sqft,Ready to Move,5 out of 14,Resale,Semi-Furnished,East,Garden/Park,Vijay Enclave,2,2,NaN,Co-operative Society,NaN,NaN,NaN,NaN
3,2 BHK Ready to Occupy Flat for sale in Charany...,Have a look at this immaculate 2 BHK flat for ...,bangalore,NaN,Ready to Move,1 out of 3,Resale,Semi-Furnished,NaN,NaN,NaN,2,NaN,NaN,NaN,1100 sqft,NaN,NaN,87.0 USD
4,2 BHK Ready to Occupy Flat for sale Kalyan East,"Kalyan East, Thane has an appealing 2 BHK flat...",thane,NaN,Ready to Move,3 out of 4,Resale,Semi-Furnished,NaN,NaN,NaN,4,NaN,NaN,NaN,1100 sqft,NaN,NaN,48.0 USD


We’ll begin by calculating the average price of all the houses. It’s not the most accurate method, but it gives us a baseline and helps you understand how the submissions work.

In [4]:
train_df['Amount_USD'].mean()

np.float64(135075.36967741934)

The **y_pred** using the average price will be:

In [5]:
baseline = train_df['Amount_USD'].mean()

In [6]:
baseline

np.float64(135075.36967741934)

Number of rows in our **y_true** are 6519. So, our **y_pred** will be

In [7]:
y_pred = [baseline] * 6344

We have our `y_pred` ready. Now! Go ahead and try the checking if the activity passes.

👈👈👈 Click on `Check Activity` to see the result.

Your turn now — feel free to delete the earlier code before continuing.

---

#### Your code:

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def _to_int_from_str(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    s = str(val).strip()
    s = s.replace('>', '').replace('+', '')
    s = re.sub(r'\D', '', s)
    return int(s) if s != '' else np.nan

def convert_carpet_area(area):
    try:
        if pd.isna(area) or str(area).strip() == '':
            return np.nan
        if isinstance(area, str):
            a = area.lower().strip()
            if 'sqft' in a:
                a = a.replace('sqft', '').strip()
                return float(a)
            if 'sqm' in a:
                a = a.replace('sqm', '').strip()
                return float(a) * 10.7639
            return float(re.sub(r'[^\d.]', '', a))
        return float(area)
    except Exception:
        return np.nan

def str_usd_to_float(s):
    if pd.isna(s) or str(s).strip() == '':
        return np.nan
    t = str(s).replace('USD', '').strip()
    t = re.sub(r'[^\d\.]', '', t)
    try:
        return float(t) if t != '' else np.nan
    except:
        return np.nan

def initial_clean(df, drop_na_bathroom=True):
    df = df.copy()
    drop_cols = ['Society','overlooking', 'Car Parking', 'Super Area', 'Dimensions', 'Plot Area']
    present = [c for c in drop_cols if c in df.columns]
    if present:
        df = df.drop(columns=present)
    if drop_na_bathroom and 'Bathroom' in df.columns:
        df = df.dropna(subset=['Bathroom']).copy()
    if 'Balcony' in df.columns:
        df['Balcony'] = df['Balcony'].fillna(0)
    for col in ('Bathroom', 'Balcony'):
        if col in df.columns:
            df[col] = df[col].apply(_to_int_from_str).astype('Int64')
    for c in ['facing', 'Ownership']:
        if c in df.columns:
            df[c] = df[c].fillna('Unknown')
    if 'Carpet Area' in df.columns:
        df['Carpet Area'] = df['Carpet Area'].apply(convert_carpet_area)
        df['Carpet Area'] = df['Carpet Area'].fillna(0)
    if 'Price_USD' in df.columns:
        df['Price_USD'] = df['Price_USD'].apply(str_usd_to_float)
    return df

train_df = initial_clean(train_df, drop_na_bathroom=True)

cols_to_fill_mode = [c for c in ['Status', 'Transaction', 'Furnishing'] if c in train_df.columns]
for col in cols_to_fill_mode:
    if train_df[col].isna().any():
        mode_val = train_df[col].mode().iloc[0]
        train_df[col].fillna(mode_val, inplace=True)

drop_later = [c for c in ['Title', 'Description', 'Floor', 'Status'] if c in train_df.columns]
train_df = train_df.drop(columns=drop_later)

# Fill Amount_USD NaN with mean
if 'Amount_USD' in train_df.columns:
    mean_amount = train_df['Amount_USD'].mean()
    train_df['Amount_USD'].fillna(mean_amount, inplace=True)

y = train_df['Amount_USD']
X = train_df.drop(columns=['Amount_USD'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4)

numeric_cols = [c for c in ['Carpet Area', 'Bathroom', 'Balcony', 'Price_USD'] if c in X_train.columns]
extra_numeric = [c for c in X_train.select_dtypes(include=['int64','float64','Int64']).columns if c not in numeric_cols and c != 'Amount_USD']
numeric_cols = numeric_cols + extra_numeric
categorical_cols = [c for c in ['location','Transaction', 'Furnishing', 'facing', 'Ownership'] if c in X_train.columns]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(r2)

test_df = pd.read_csv('data/test_inputs.csv')
test_df_clean = initial_clean(test_df, drop_na_bathroom=False)

for c in drop_later:
    if c in test_df_clean.columns:
        test_df_clean = test_df_clean.drop(columns=[c])
for c in numeric_cols + categorical_cols:
    if c not in test_df_clean.columns:
        test_df_clean[c] = np.nan

y_pred = pipeline.predict(test_df_clean)

y_pred
